# Desempenho e GPU

[Abrir este capítulo no Google Colab](https://colab.research.google.com/github/vnikoofard/calculo-numerico-livro/blob/main/livro/B-desempenho-gpu.ipynb)

> **Objetivos.** Ao final deste apêndice você será capaz de:
> 1. estimar o custo de um cálculo contando operações e reconhecer a complexidade dos algoritmos mais comuns;
> 2. explicar por que laços em Python são lentos e acelerá-los por vetorização, por processamento em lote e por compilação (Numba);
> 3. decidir quando uma GPU (via PyTorch) vale a pena, e em que precisão;
> 4. medir tempos de execução de forma confiável;
> 5. verificar que a versão rápida de um código dá a **mesma resposta** que a lenta.

Os tempos deste apêndice foram medidos no computador que gerou o livro, sem GPU. No seu, os números serão outros; as razões entre eles, e as conclusões, devem se manter.

In [ ]:
# Preparação do ambiente: funciona no seu computador, no Colab e no navegador (JupyterLite).
import sys
try:
    import cnlivro
except ImportError:
    if sys.platform == "emscripten":  # JupyterLite: Python rodando no navegador
        import micropip
        await micropip.install(["plotly", "scipy",
            "https://vnikoofard.github.io/calculo-numerico-livro/pacote/cnlivro-0.1.0-py3-none-any.whl"])
    else:  # Colab ou instalação local sem o pacote
        %pip install -q git+https://github.com/vnikoofard/calculo-numerico-livro

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from cnlivro.estilo import aplicar_estilo, mostrar, grafico_convergencia, grafico_iteracoes, com_controle, numero, CORES
from cnlivro import raizes
from cnlivro.verificacao import ordem_observada, ordem_iterativa, compara

aplicar_estilo()

In [ ]:
import time
import torch
from cnlivro.edo import contorno_dif_finitas
from cnlivro.desempenho import cronometrar, dispositivo, thomas_lote
try:
    import numba
    TEM_NUMBA = True
except ImportError:          # por exemplo, no navegador (JupyterLite)
    TEM_NUMBA = False
print("dispositivo do PyTorch:", dispositivo(), "| Numba disponível:", TEM_NUMBA)

## 1. Problema de engenharia: 100 mil pilhas de bagaço

No apêndice A, calculamos a probabilidade de uma pilha de bagaço pegar fogo sorteando 100 mil cenários. Lá, cada cenário custou quase nada, porque o problema de Frank-Kamenetskii tem solução exata. Em problemas reais (uma pilha com forma irregular, propriedades que variam com a umidade, uma cinética mais complicada), cada cenário exige resolver o problema de contorno numericamente: diferenças finitas e Newton, como no capítulo 14. A pergunta passa a ser de **custo**: quanto tempo levam 100 mil soluções, e como reduzir esse tempo?

O ponto de partida é o óbvio: um laço que chama, para cada cenário, a função `contorno_dif_finitas` do pacote do livro (capítulo 14), com 100 subintervalos.

In [ ]:
rng = np.random.default_rng(2026)
N = 100_000
E_R, T_ref = 11_000.0, 303.15
L = rng.uniform(2.2, 2.8, N)
T0 = rng.normal(30, 3, N) + 273.15
k = rng.normal(0.08, 0.012, N)
a = np.exp(rng.normal(0, np.log(2) / 1.96, N))
deltas = 0.45 * (L / 2.5) ** 2 * (0.08 / k) * a * (T_ref / T0) ** 2 * np.exp(E_R * (1 / T_ref - 1 / T0))
delta_c = 0.8784576797812903          # limite exato (apêndice A): acima dele, não há solução

def cenario(d):
    r = contorno_dif_finitas(lambda x, y, dy: -d * np.exp(y), (-1, 1), 0.0, ("dirichlet", 0.0), 100)
    return r.sucesso

n_teste = 200
t0 = time.perf_counter()
fogo_laco = np.array([not cenario(d) for d in deltas[:n_teste]])
t_laco = (time.perf_counter() - t0) / n_teste
print(f"{1e3 * t_laco:.1f} ms por cenário  ->  {N * t_laco / 60:.1f} min para {N} cenários")
print(f"nos {n_teste} primeiros: {fogo_laco.sum()} sem solução; pelo critério exato δ > δ_c: {(deltas[:n_teste] > delta_c).sum()}")

A resposta está certa (os cenários sem solução são exatamente os com $\delta > \delta_c$), mas o custo, multiplicado por 100 mil, fica na casa dos minutos. Num estudo de projeto, com dezenas de alternativas, ou num modelo 2D, isso vira horas. Vamos ver de onde vem o tempo e como reduzi-lo.

## 2. A ideia em uma imagem

O Python é uma linguagem **interpretada**: cada operação de um laço passa pelo interpretador, que descobre os tipos, procura a operação certa e cria um objeto novo para o resultado. Isso custa dezenas de nanossegundos por operação, enquanto o processador faz uma soma de números em menos de um nanossegundo. Uma operação do NumPy sobre um vetor paga esse preço **uma vez** e depois percorre os dados num laço compilado. Somando $n$ números das duas maneiras:

In [ ]:
ns_v = np.unique(np.logspace(1, 6.5, 14).astype(int))
t_py, t_np = [], []
for n in ns_v:
    lista = list(np.random.default_rng(0).random(n))
    vetor = np.array(lista)
    def soma_laco(v=lista):
        s = 0.0
        for x in v:
            s += x
        return s
    rep = 3 if n > 100_000 else 20
    t_py.append(cronometrar(soma_laco, repeticoes=rep).minimo)       # o mínimo filtra interrupções do sistema
    t_np.append(cronometrar(np.sum, vetor, repeticoes=rep * 5).minimo)
t_py, t_np = np.array(t_py), np.array(t_np)
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.12, subplot_titles=["tempo total", "tempo por elemento"])
for col, f in [(1, lambda t: t), (2, lambda t: t / ns_v * 1e9)]:
    fig.add_trace(go.Scatter(x=ns_v, y=f(t_py), mode="lines+markers", name="laço em Python", showlegend=(col == 1),
                             line=dict(color=CORES["erro"], width=3)), 1, col)
    fig.add_trace(go.Scatter(x=ns_v, y=f(t_np), mode="lines+markers", name="np.sum (laço compilado)", showlegend=(col == 1),
                             line=dict(color=CORES["aproximacao"], width=3)), 1, col)
    fig.update_xaxes(title="número de elementos n", type="log", exponentformat="power", row=1, col=col)
fig.update_yaxes(title="tempo (s)", type="log", exponentformat="power", row=1, col=1)
fig.update_yaxes(title="ns por elemento", type="log", dtick=1, row=1, col=2)
razao = t_py[-1] / t_np[-1]
fig.update_layout(width=980, height=430, margin=dict(t=110), legend=dict(orientation="h", y=-0.22),
                  title=f"Somar n números: com n grande, o laço em Python é {razao:.0f} vezes mais lento")
mostrar(fig)

**O que notar:** à esquerda, as duas retas têm inclinação 1 (custo proporcional a $n$), mas estão separadas por um fator constante e grande. À direita, o tempo por elemento do laço em Python fica parado em dezenas de nanossegundos, enquanto o do NumPy cai até abaixo de um nanossegundo quando $n$ cresce: para vetores pequenos, o custo fixo da chamada (alguns microssegundos) domina. Toda a aceleração deste apêndice vem de duas ideias: **tirar os laços do interpretador** e **dar a cada chamada trabalho suficiente** para pagar o custo fixo.

## 3. Formulação

### 3.1 Onde o tempo vai

Ordens de grandeza aproximadas num computador atual:

| Operação | Tempo típico |
| --- | --- |
| uma operação de ponto flutuante num núcleo (com instruções vetoriais, várias por ciclo) | 0,1 a 1 ns |
| uma operação num laço do Python | 30 a 100 ns |
| custo fixo de uma chamada do NumPy | 1 a 5 µs |
| ler 1 GB da memória principal | 0,02 a 0,1 s |
| custo fixo de uma operação na GPU (lançar um *kernel*) | 5 a 20 µs |
| copiar 1 GB entre a memória do computador e a da GPU | 0,05 a 0,1 s |

Há dois limites para um cálculo bem escrito: o número de operações (**limitado por processamento**) e a quantidade de dados lidos e escritos (**limitado pela memória**). Somar dois vetores faz uma operação para cada três números movidos: é limitado pela memória. Multiplicar matrizes faz $2n^3$ operações sobre $3n^2$ números: é limitado por processamento, e por isso atinge o pico da máquina.

### 3.2 Complexidade

Contar operações em função do tamanho do problema diz como o custo **cresce**, que é o que importa para decidir se um cálculo é viável:

| Algoritmo | Operações | Capítulo |
| --- | --- | --- |
| produto escalar, soma de vetores | $\sim n$ | — |
| sistema tridiagonal (Thomas) | $\sim 8n$ | 4, 14 |
| produto matriz-vetor (denso) | $\sim 2n^2$ | 4 |
| fatoração LU (densa) | $\sim \tfrac23 n^3$ | 4 |
| produto de matrizes | $\sim 2n^3$ | — |
| FFT | $\sim 5n\log_2 n$ | — |
| sistema esparso 2D (malha $m \times m$, $n = m^2$), direto | $\sim n^{1{,}5}$ | 15 |
| explícito para o calor 2D até um tempo fixo | $\sim n^2$ (passos $\propto n$) | 16 |

A lição mais importante deste apêndice está nessa tabela: **trocar de algoritmo costuma valer mais que trocar de máquina**. Uma GPU 50 vezes mais rápida não compensa usar LU densa ($n^3$) num sistema tridiagonal que Thomas resolve em $8n$ operações: com $n = 10^4$, a diferença é de um fator de $10^7$.

### 3.3 Vetorização e processamento em lote

**Vetorizar** é reescrever o cálculo como operações sobre vetores inteiros, para que os laços rodem dentro do NumPy (ou do PyTorch). Quando o problema é **muitos problemas pequenos e independentes**, como os 100 mil cenários, a vetorização natural é **em lote**: em vez de um laço sobre os cenários, com um sistema de 99 incógnitas em cada um, escrevemos o algoritmo uma vez, operando sobre um arranjo de forma (cenários × incógnitas). O laço que sobra é sobre as 99 incógnitas (o algoritmo de Thomas é sequencial ao longo delas), e cada passo dele age sobre os 100 mil cenários de uma vez. A função `thomas_lote` do pacote faz isso.

### 3.4 Compilação: Numba

Algoritmos com dependências sequenciais (recorrências como a de Thomas, iterações com testes de parada diferentes em cada cenário) são mais naturais como laços. O **Numba** compila uma função Python, escrita só com NumPy e laços, para código de máquina, na primeira vez em que ela é chamada (compilação *just-in-time*):

```python
from numba import njit

@njit
def soma(v):
    s = 0.0
    for x in v:
        s += x
    return s
```

A primeira chamada inclui a compilação (de décimos de segundo a alguns segundos); as seguintes rodam na velocidade de C. O Numba aceita só um subconjunto do Python (números, arranjos do NumPy, laços, funções matemáticas), o que é exatamente o que um núcleo numérico precisa.

### 3.5 GPU com PyTorch

Uma GPU tem milhares de unidades de cálculo e uma memória muito rápida, e executa bem a **mesma operação sobre muitos dados**. Com o PyTorch, o mesmo código roda na CPU ou na GPU, trocando o dispositivo dos tensores (`torch.tensor(..., device="cuda")`). Ela vale a pena quando:

- cada operação age sobre muitos dados (da ordem de $10^5$ a $10^6$ números ou mais), para diluir o custo fixo de cada *kernel*;
- os dados **ficam** na GPU durante muitas operações, porque a cópia entre as memórias é lenta;
- a precisão necessária é compatível com o hardware: GPUs de uso geral fazem contas em `float32` dezenas de vezes mais depressa que em `float64` (só as de centro de dados têm `float64` rápido).

Ela não ajuda em problemas pequenos, em algoritmos sequenciais (como o laço de Thomas, cujos 99 passos têm de ser executados um depois do outro) nem em códigos que copiam dados a cada passo.

**Precisão.** Em `float32`, o épsilon da máquina é $\varepsilon \approx 1{,}2\times10^{-7}$ (capítulo 2), contra $2{,}2\times10^{-16}$ em `float64`. Toda tolerância, todo critério de parada e todo resultado que dependa de cancelamentos têm de ser revistos ao mudar de precisão.

### 3.6 Medir direito, e a lei de Amdahl

Para medir o tempo de um trecho de código: use `time.perf_counter`; repita várias vezes e use a **mediana**; descarte a primeira execução (compilação, alocação de memória, *caches*); na GPU, espere o fim dos cálculos (`torch.cuda.synchronize()`), porque as operações são **assíncronas**. A função `cronometrar` do pacote faz tudo isso.

Antes de acelerar, meça **onde** o tempo é gasto. Se uma fração $s$ do tempo não é acelerada e o resto fica $p$ vezes mais rápido, o ganho total é

$$
\text{aceleração} = \frac{1}{s + (1 - s)/p} \le \frac1s \qquad \text{(lei de Amdahl)}.
$$

Acelerar infinitamente 90 % do código dá no máximo um ganho de 10.

## 4. À mão

**Custo de um cenário.** Cada iteração de Newton, com $m = 99$ incógnitas, calcula o resíduo (umas 6 operações por nó, mais uma exponencial), a jacobiana tridiagonal (2 por nó) e resolve o sistema por Thomas (8 por nó): umas 20 operações por nó, ou 2 000 por iteração. Com umas 6 iterações, são cerca de $10^4$ operações por cenário e $10^9$ para 100 mil cenários: **um segundo** de processador, em código compilado. Em Python puro, a 50 ns por operação, seriam 50 segundos. O laço da seção 1 gasta ainda mais, porque a função do pacote é genérica (monta a jacobiana por diferenças finitas e faz verificações).

**Amdahl.** Se a montagem dos 100 mil cenários, em Python, levar 0,2 s e o resto 10 s, acelerar o resto 100 vezes dá $10{,}2/(0{,}2 + 0{,}1) = 34$ vezes, e não 100.

**Memória.** Guardar a solução de 100 mil cenários com 99 incógnitas em `float64` ocupa $10^5 \times 99 \times 8 \approx 80$ MB. Uma matriz **densa** de 99 × 99 por cenário, para usar um solver denso em lote, ocuparia $10^5 \times 99^2 \times 8 \approx 7{,}8$ GB.

## 5. Explore

### 5.1 Complexidade medida

O tempo para resolver $A\mathbf x = \mathbf b$ com $A$ tridiagonal de ordem $n$ por três caminhos: LU densa (`np.linalg.solve`), solver de banda do SciPy (`solve_banded`) e Thomas em Python puro.

In [ ]:
from scipy.linalg import solve_banded

def thomas_py(a, b, c, d):
    n = len(b)
    cp, dp = [0.0] * n, [0.0] * n
    cp[0], dp[0] = c[0] / b[0], d[0] / b[0]
    for i in range(1, n):
        m = b[i] - a[i] * cp[i - 1]
        cp[i], dp[i] = c[i] / m, (d[i] - a[i] * dp[i - 1]) / m
    x = [0.0] * n
    x[-1] = dp[-1]
    for i in range(n - 2, -1, -1):
        x[i] = dp[i] - cp[i] * x[i + 1]
    return x

ns_c = 100 * 2 ** np.arange(11)                   # 100 a 102 400
n_denso = 6                                       # a LU densa só até n = 3200
tempos = {"LU densa (np.linalg.solve)": [], "banda (solve_banded)": [], "Thomas em Python puro": []}
for j, n in enumerate(ns_c):
    a_, b_, c_, d_ = np.ones(n), np.full(n, -2.5), np.ones(n), np.random.default_rng(1).random(n)
    if j < n_denso:
        A = np.diag(b_) + np.diag(a_[1:], -1) + np.diag(c_[:-1], 1)
        tempos["LU densa (np.linalg.solve)"].append(cronometrar(np.linalg.solve, A, d_, repeticoes=3).minimo)
    ab = np.vstack([np.r_[0, c_[:-1]], b_, np.r_[a_[1:], 0]])
    tempos["banda (solve_banded)"].append(cronometrar(solve_banded, (1, 1), ab, d_, repeticoes=10).minimo)
    tempos["Thomas em Python puro"].append(cronometrar(thomas_py, list(a_), list(b_), list(c_), list(d_), repeticoes=3).minimo)
fig = go.Figure()
for (nome, t), cor in zip(tempos.items(), [CORES["erro"], CORES["aproximacao"], CORES["secundaria"]]):
    t = np.array(t)
    nn = ns_c[:t.size]
    incl = np.polyfit(np.log(nn[-3:]), np.log(t[-3:]), 1)[0]     # inclinação nos três maiores n
    fig.add_trace(go.Scatter(x=nn, y=t, mode="lines+markers", name=f"{nome}: inclinação {numero(incl, 1)}",
                             line=dict(color=cor, width=3)))
fig.update_xaxes(title="ordem do sistema n", type="log", dtick=1, exponentformat="power")
fig.update_yaxes(title="tempo (s)", type="log", dtick=1, exponentformat="power")
fig.update_layout(width=800, height=450, legend=dict(orientation="h", y=-0.22),
                  title=f"Sistema tridiagonal: com n = {ns_c[n_denso - 1]}, a LU densa é "
                        f"{tempos['LU densa (np.linalg.solve)'][-1] / tempos['banda (solve_banded)'][n_denso - 1]:.0f} vezes mais lenta que a de banda")
mostrar(fig)

**O que notar:** nos maiores $n$, as inclinações no gráfico log-log se aproximam das complexidades: entre 2 e 3 para a LU densa (o expoente 3 só aparece para $n$ ainda maior, quando as contas dominam o acesso à memória) e perto de 1 para os outros dois. Para $n$ pequeno, as curvas ficam mais planas, porque o custo fixo da chamada pesa mais que as contas. O Thomas em Python puro ganha da LU densa compilada em todos os tamanhos, mesmo com ela usando a biblioteca de álgebra linear otimizada e vários núcleos. A diferença entre a banda compilada e o Thomas em Python é o fator constante do interpretador da seção 2.

### 5.2 Lote: quantos cenários por chamada?

A mesma solução de sistemas tridiagonais de 99 incógnitas, agora em lote, com `thomas_lote` (NumPy). O tempo **por sistema** em função do tamanho do lote:

In [ ]:
def thomas_linhas(a, b, c, d):
    # o mesmo algoritmo, percorrendo as colunas de arranjos (lote, n): saltos de n números na memória
    n = b.shape[1]
    cp, dp = np.empty_like(b), np.empty_like(b)
    cp[:, 0], dp[:, 0] = c[:, 0] / b[:, 0], d[:, 0] / b[:, 0]
    for i in range(1, n):
        m = b[:, i] - a[:, i] * cp[:, i - 1]
        cp[:, i], dp[:, i] = c[:, i] / m, (d[:, i] - a[:, i] * dp[:, i - 1]) / m
    x = np.empty_like(b)
    x[:, -1] = dp[:, -1]
    for i in range(n - 2, -1, -1):
        x[:, i] = dp[:, i] - cp[:, i] * x[:, i + 1]
    return x

lotes = np.array([1, 10, 100, 1000, 10_000, 100_000])
t_lote, t_linhas = [], []
for B in lotes:
    aa, cc = np.ones((B, 99)), np.ones((B, 99))
    bb, dd = np.full((B, 99), -2.5), np.random.default_rng(2).random((B, 99))
    rep = 3 if B > 1000 else 10
    t_lote.append(cronometrar(thomas_lote, aa, bb, cc, dd, repeticoes=rep).minimo / B)
    t_linhas.append(cronometrar(thomas_linhas, aa, bb, cc, dd, repeticoes=rep).minimo / B)
t_lote, t_linhas = np.array(t_lote), np.array(t_linhas)
fig = go.Figure()
fig.add_trace(go.Scatter(x=lotes, y=t_linhas * 1e6, mode="lines+markers", name="colunas de um arranjo (lote, n): com saltos",
                         line=dict(color=CORES["erro"], width=3, dash="dash")))
fig.add_trace(go.Scatter(x=lotes, y=t_lote * 1e6, mode="lines+markers", name="thomas_lote: arranjo (n, lote), contíguo",
                         line=dict(color=CORES["aproximacao"], width=3)))
fig.update_xaxes(title="sistemas por chamada (tamanho do lote)", type="log", dtick=1, exponentformat="power")
fig.update_yaxes(title="tempo por sistema (µs)", type="log", dtick=1)
fig.update_layout(width=800, height=450, legend=dict(orientation="h", y=-0.22),
                  title=f"Um sistema por vez: {numero(t_lote[0] * 1e6, 0)} µs; em lote: {numero(t_lote[1:].min() * 1e6, 1)} µs "
                        f"(com saltos na memória, {numero(t_linhas[-1] * 1e6, 1)} µs para 100 mil)")
mostrar(fig)

**O que notar:** com um sistema por chamada, o laço de 99 passos paga o custo fixo do NumPy umas mil vezes, para fazer poucas contas. Com lotes grandes, o mesmo custo fixo se divide entre milhares de sistemas, e o tempo por sistema cai por um fator de centenas. É a mesma curva da seção 2, agora com um algoritmo de verdade.

A curva tracejada mostra um efeito que a contagem de operações não vê: a **organização na memória**. O NumPy guarda um arranjo linha por linha. Com os sistemas nas linhas, forma (lote, n), a "coluna" `b[:, i]` que cada passo de Thomas lê está espalhada pela memória, com saltos de 99 números entre um elemento e o seguinte. Enquanto os dados cabem nas memórias rápidas do processador (os *caches*, de alguns megabytes), isso pouco importa; com 100 mil sistemas, cada arranjo tem 80 MB, e cada passo passa a esperar pela memória principal. A função `thomas_lote` copia os arranjos para a forma (n, lote) antes de começar, o que deixa cada coluna contígua: as mesmas contas, cerca de duas vezes mais rápidas para os lotes grandes, mesmo pagando as cópias. As duas curvas ainda sobem depois de mil sistemas por chamada, porque os arranjos deixam de caber no *cache* e cada passo lê e escreve na memória principal. Na prática, para lotes muito grandes compensa dividir o trabalho em blocos de alguns milhares de sistemas.

## 6. Trabalhando com o agente

### 6.1 O pedido e o código

> *"Acelere o Monte Carlo da pilha de bagaço: para cada um dos cenários (vetor `deltas`), resolva θ'' = −δ e^θ, θ(±1) = 0, por diferenças finitas com 100 subintervalos e Newton, e conte os cenários em que Newton não converge (pilha sem equilíbrio). Use a GPU se houver."*

Um código típico (aqui, para os primeiros 20 mil cenários):

In [ ]:
# Código devolvido pelo agente (copiado sem alterações)
import torch

dev = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float32              # float32: muito mais rápido na GPU

def thomas_torch(a, b, c, d):
    n = b.shape[1]
    cp, dp = torch.empty_like(b), torch.empty_like(b)
    cp[:, 0], dp[:, 0] = c[:, 0] / b[:, 0], d[:, 0] / b[:, 0]
    for i in range(1, n):
        m = b[:, i] - a[:, i] * cp[:, i - 1]
        cp[:, i], dp[:, i] = c[:, i] / m, (d[:, i] - a[:, i] * dp[:, i - 1]) / m
    x = torch.empty_like(b)
    x[:, -1] = dp[:, -1]
    for i in range(n - 2, -1, -1):
        x[:, i] = dp[:, i] - cp[:, i] * x[:, i + 1]
    return x

def newton_lote(delta, n=100, tol=1e-8, max_iter=30):
    h, m = 2.0 / n, n - 1
    d = torch.as_tensor(delta, dtype=dtype, device=dev)[:, None]
    B = d.shape[0]
    th = torch.zeros((B, m), dtype=dtype, device=dev)
    um = torch.ones((B, m), dtype=dtype, device=dev)
    convergiu = torch.zeros(B, dtype=torch.bool, device=dev)
    for _ in range(max_iter):
        tp = torch.nn.functional.pad(th, (1, 1))                     # θ = 0 nas pontas
        R = (tp[:, 2:] - 2 * th + tp[:, :-2]) / h**2 + d * torch.exp(th)
        dx = thomas_torch(um / h**2, -2 / h**2 + d * torch.exp(th), um / h**2, -R)
        th = torch.where(convergiu[:, None], th, th + dx)
        convergiu |= dx.abs().max(dim=1).values < tol
        if convergiu.all():
            break
    return convergiu & torch.isfinite(th).all(dim=1), th[:, m // 2]

n_ag = 20_000
t0 = time.perf_counter()
ok, th0 = newton_lote(deltas[:n_ag])
t_agente = time.perf_counter() - t0
print(f"{n_ag} cenários em {t_agente:.2f} s ({dev}, {dtype})")
print(f"probabilidade de fogo: {100 * (1 - ok.float().mean().item()):.1f} %")

O código é bem pensado: o algoritmo de Thomas em lote, com o laço sobre as incógnitas e não sobre os cenários; os cenários que já convergiram deixam de ser atualizados; tudo fica no dispositivo, sem cópias. Os 20 mil cenários levam poucos segundos, mesmo sem GPU. Mas a probabilidade de fogo saiu perto de **86 %**, e o apêndice A deu 10,5 %.

### 6.2 Verificação

**A mesma resposta que a versão lenta.** Toda otimização tem de ser verificada contra a versão de referência, nos mesmos dados. Aqui temos até um critério exato ($\delta > \delta_c$):

In [ ]:
fogo_ag = ~ok.cpu().numpy()
fogo_exato = deltas[:n_ag] > delta_c
print(f"fogo segundo o código do agente: {fogo_ag.sum()} cenários; pelo critério exato: {fogo_exato.sum()}")
print(f"cenários classificados de modo diferente: {np.sum(fogo_ag != fogo_exato)}")
print(f"nos 200 primeiros, o laço da seção 1 concorda com o exato: {np.all(fogo_laco == fogo_exato[:200])}; "
      f"o código do agente: {np.all(fogo_ag[:200] == fogo_exato[:200])}")

**O que o Newton está fazendo nos cenários "sem solução"?** Olhando o tamanho da correção, iteração por iteração, num cenário estável ($\delta = 0{,}3$), nas duas precisões:

In [ ]:
def historico_newton(delta, tipo, n=100, iteracoes=12):
    h, m = 2.0 / n, n - 1
    d = torch.tensor([[delta]], dtype=tipo)
    th = torch.zeros((1, m), dtype=tipo)
    um = torch.ones((1, m), dtype=tipo)
    hist = []
    for _ in range(iteracoes):
        tp = torch.nn.functional.pad(th, (1, 1))
        R = (tp[:, 2:] - 2 * th + tp[:, :-2]) / h**2 + d * torch.exp(th)
        dx = thomas_torch(um / h**2, -2 / h**2 + d * torch.exp(th), um / h**2, -R)
        th = th + dx
        hist.append(dx.abs().max().item())
    return hist

for tipo in (torch.float64, torch.float32):
    print(f"{str(tipo):14s}", " ".join(f"{v:.0e}" for v in historico_newton(0.3, tipo)))
print(f"épsilon da máquina: float32 {torch.finfo(torch.float32).eps:.1e}, float64 {torch.finfo(torch.float64).eps:.1e}")

Em `float64`, a correção cai quadraticamente até $10^{-17}$ e passa pela tolerância de $10^{-8}$ na quinta iteração. Em `float32`, ela cai até uns $10^{-7}$ e **para**: abaixo disso, o resíduo calculado é só ruído de arredondamento, e cada "correção" apenas troca um erro de arredondamento por outro. A tolerância de $10^{-8}$ nunca é atingida, e o código classifica o cenário como "Newton não convergiu", ou seja, fogo.

### 6.3 Por quê

A tolerância de $10^{-8}$ é razoável em `float64` e **impossível** em `float32`. O resíduo é a diferença entre termos da ordem de $\theta/h^2 \sim 10^3$, calculados com erro relativo $\varepsilon_{32} \approx 10^{-7}$: o seu "zero" numérico é da ordem de $10^{-4}$, e a correção correspondente, de $10^{-7}$ (capítulo 2: o limite de precisão de um cálculo é o épsilon vezes o tamanho dos termos que se cancelam). Os 14 % de cenários que o código do agente deu como estáveis são os poucos em que, por sorte, uma correção caiu abaixo de $10^{-8}$. A troca de precisão, feita "porque é mais rápido", mudou a resposta de 10 % para 86 %, sem nenhum erro ou aviso.

### 6.4 A correção

Duas correções, e a medida honesta do ganho de cada versão:

- `float64` com a mesma tolerância (na CPU, ou numa GPU de centro de dados);
- ou `float32` com uma tolerância compatível com a precisão, por exemplo $10^{-5}$, verificando a resposta contra `float64` numa amostra.

In [ ]:
def newton_lote_tipo(delta, tipo, tol):
    global dtype
    dtype_antes, dtype = dtype, tipo
    try:
        return newton_lote(delta, tol=tol)
    finally:
        dtype = dtype_antes

for tipo, tol in [(torch.float64, 1e-8), (torch.float32, 1e-5)]:
    r = cronometrar(newton_lote_tipo, deltas[:n_ag], tipo, tol, repeticoes=3)
    ok_t = r.valor[0].cpu().numpy()
    print(f"{str(tipo):14s} tol {tol:.0e}: P(fogo) = {100 * (1 - ok_t.mean()):.2f} %, "
          f"diferenças para o exato: {np.sum(~ok_t != fogo_exato)}, {r.mediana:.2f} s")
print(f"exato (δ > δ_c): {100 * fogo_exato.mean():.2f} %")

As duas versões corrigidas classificam todos os cenários como o critério exato (ou erram por um ou dois, colados em $\delta_c$). Agora, o ganho de desempenho. Para completar, uma versão compilada com Numba, que resolve um cenário de cada vez num laço compilado (a forma natural para Thomas e para o teste de parada individual):

In [ ]:
if TEM_NUMBA:
    from numba import njit

    @njit(cache=False)
    def newton_numba(deltas, n=100, tol=1e-8, max_iter=30):
        h2 = (2.0 / n) ** 2
        m = n - 1
        ok = np.zeros(deltas.size, dtype=np.bool_)
        th = np.zeros(m)
        R, diag, cp, dp, dx = np.zeros(m), np.zeros(m), np.zeros(m), np.zeros(m), np.zeros(m)
        for j in range(deltas.size):
            d = deltas[j]
            th[:] = 0.0
            for _ in range(max_iter):
                for i in range(m):
                    esq = th[i - 1] if i > 0 else 0.0
                    dir_ = th[i + 1] if i < m - 1 else 0.0
                    e = np.exp(th[i])
                    R[i] = (esq - 2 * th[i] + dir_) / h2 + d * e
                    diag[i] = -2 / h2 + d * e
                # Thomas, com sub e superdiagonal 1/h²
                cp[0], dp[0] = (1 / h2) / diag[0], -R[0] / diag[0]
                for i in range(1, m):
                    den = diag[i] - (1 / h2) * cp[i - 1]
                    cp[i], dp[i] = (1 / h2) / den, (-R[i] - (1 / h2) * dp[i - 1]) / den
                dx[m - 1] = dp[m - 1]
                for i in range(m - 2, -1, -1):
                    dx[i] = dp[i] - cp[i] * dx[i + 1]
                passo = 0.0
                for i in range(m):
                    th[i] += dx[i]
                    passo = max(passo, abs(dx[i]))
                if not np.all(np.isfinite(th)):      # max(0.0, nan) dá 0.0: o nan tem de ser testado à parte
                    break
                if passo < tol:
                    ok[j] = True
                    break
        return ok

    t0 = time.perf_counter()
    newton_numba(deltas[:10])
    t_compila = time.perf_counter() - t0
    r_nb = cronometrar(newton_numba, deltas[:n_ag], repeticoes=3)
    print(f"Numba: compilação {t_compila:.2f} s; {n_ag} cenários em {r_nb.mediana:.3f} s; "
          f"diferenças para o exato: {np.sum(~r_nb.valor != fogo_exato)}")
    r_nb_tudo = cronometrar(newton_numba, deltas, repeticoes=1)
    print(f"Numba, todos os {N} cenários: {r_nb_tudo.mediana:.2f} s, P(fogo) = {100 * (1 - r_nb_tudo.valor.mean()):.2f} % "
          f"(exato: {100 * np.mean(deltas > delta_c):.2f} %)")
else:
    print("Numba não está disponível neste ambiente (por exemplo, no navegador).")

Resumindo, para 20 mil cenários neste computador: o laço original levaria perto de um minuto; o lote em PyTorch na CPU, alguns segundos (e o `float32` quase não ganha do `float64` na CPU); o Numba, uma fração de segundo, e todos os 100 mil cenários levam cerca de um segundo, como na estimativa à mão da seção 4. O Numba ganha do lote aqui porque cada cenário para assim que converge e porque os 99 valores de um cenário cabem no *cache*; o lote percorre arranjos grandes a cada passo de Thomas (seção 5.2). Numa GPU, o lote em `float32` com a tolerância corrigida tende a ganhar do lote na CPU, mas o laço de 99 passos sequenciais de Thomas limita o ganho (seção 3.5), e o Numba na CPU continua competitivo. A escolha depende do problema, e só se decide **medindo**.

> **Lição.** Uma versão rápida só vale se der a **mesma resposta** que a lenta: compare as duas nos mesmos dados antes de comemorar o ganho. Trocar `float64` por `float32` (ou rodar na GPU, que favorece `float32`) exige rever **todas** as tolerâncias e critérios de parada: com $\varepsilon \approx 10^{-7}$, nenhum resíduo ou correção desce muito abaixo de $10^{-7}$ vezes a escala dos termos. E meça o tempo direito (mediana, sem a compilação, com sincronização na GPU) antes de escolher entre lote, compilação e GPU.

## 7. Quando falha

### 7.1 Medir a compilação junto

A primeira chamada de uma função do Numba (ou do `torch.compile`) inclui a compilação. Medindo só uma vez:

In [ ]:
if TEM_NUMBA:
    @njit
    def soma_nb(v):
        s = 0.0
        for x in v:
            s += x
        return s

    v = np.random.default_rng(3).random(1_000_000)
    t0 = time.perf_counter(); soma_nb(v); t_primeira = time.perf_counter() - t0
    r = cronometrar(soma_nb, v, repeticoes=7)
    r_np = cronometrar(np.sum, v, repeticoes=7)
    print(f"Numba, primeira chamada: {1e3 * t_primeira:.1f} ms;  mediana das seguintes: {1e3 * r.mediana:.2f} ms;  "
          f"np.sum: {1e3 * r_np.mediana:.2f} ms")
else:
    print("Numba não está disponível neste ambiente.")

Medida uma vez, a versão compilada parece dezenas de vezes mais lenta do que é; depois de compilada, ela fica na mesma faixa do `np.sum` (que, além de compilado, usa instruções vetoriais). O erro oposto também é comum: medir só a segunda chamada e esquecer que, num programa que chama a função uma única vez, a compilação é paga. Na GPU, o equivalente é medir sem `torch.cuda.synchronize()`: o relógio para quando as operações foram **enfileiradas**, não quando terminaram, e a GPU parece absurdamente rápida.

### 7.2 Vetorizar demais

A vetorização pode trocar tempo por memória sem que se perceba. Uma tentação comum, quando há uma biblioteca de álgebra linear em lote, é montar a matriz **densa** de cada cenário e chamar um solver denso em lote, em vez de escrever Thomas:

In [ ]:
def solve_denso_lote(B, m=99):
    A = (torch.diag(torch.full((m,), -2.5, dtype=torch.float64)) + torch.diag(torch.ones(m - 1, dtype=torch.float64), 1)
         + torch.diag(torch.ones(m - 1, dtype=torch.float64), -1)).expand(B, m, m).contiguous()
    b = torch.ones((B, m, 1), dtype=torch.float64)
    return torch.linalg.solve(A, b)

for B in [1000, 4000]:
    r_d = cronometrar(solve_denso_lote, B, repeticoes=3)
    r_t = cronometrar(thomas_lote, np.ones((B, 99)), np.full((B, 99), -2.5), np.ones((B, 99)), np.ones((B, 99)), repeticoes=3)
    print(f"lote de {B}: denso {r_d.mediana:.3f} s, Thomas {r_t.mediana:.4f} s; memória das matrizes densas: "
          f"{B * 99 * 99 * 8 / 1e6:.0f} MB")
print(f"para os {N} cenários, as matrizes densas ocupariam {N * 99 * 99 * 8 / 1e9:.1f} GB")

O solver denso faz $\sim m^3$ operações por cenário, contra $8m$ de Thomas, e as matrizes densas dos 100 mil cenários não caberiam na memória de muitas GPUs. Mais rápido por operação não compensa um algoritmo pior (seção 3.2).

### 7.3 Acelerar a parte errada

Antes de acelerar, meça onde o tempo é gasto. No laço da seção 1, por exemplo, quanto do tempo de cada cenário é o Newton propriamente dito? A biblioteca padrão traz um medidor por função:

In [ ]:
import cProfile, pstats, io
perfil = cProfile.Profile()
perfil.enable()
for d in deltas[:50]:
    cenario(d)
perfil.disable()
saida = io.StringIO()
estat = pstats.Stats(perfil, stream=saida).strip_dirs().sort_stats("tottime")
estat.print_stats(6)
print("\n".join(l for l in saida.getvalue().splitlines() if l.strip()))

total = estat.total_tt
def fracao(nome, coluna):           # coluna 2: tempo próprio; 3: tempo com as chamadas internas
    return sum(v[coluna] for k, v in estat.stats.items() if k[2] == nome) / total
s_proprio, s_residuo = fracao("contorno_dif_finitas", 2), fracao("residuo", 3)
s_banda = max(v[3] for k, v in estat.stats.items() if "solve_banded" in k[2] or (k[0] == "_util.py" and k[2] == "wrapper")) / total
print(f"\ncorpo de contorno_dif_finitas: {100 * s_proprio:.0f} %;  resíduo: {100 * s_residuo:.0f} %;  solve_banded: {100 * s_banda:.0f} %")
print(f"ganho máximo acelerando só o solve_banded (Amdahl, p infinito): {1 / (1 - s_banda):.2f}")

O relatório lista as funções pelo tempo gasto nelas mesmas (`tottime`, sem contar as funções que elas chamam); `cumtime` inclui as chamadas internas. O primeiro lugar é o **corpo** de `contorno_dif_finitas`, com mais da metade do tempo: não são as contas do Newton, e sim o laço duplo em Python que monta a jacobiana de banda elemento por elemento (umas 300 atribuições por iteração), a partir das perturbações por diferenças finitas. O resíduo e a solução do sistema tridiagonal (`solve_banded`), que são o Newton propriamente dito, ficam com uma fração pequena.

Um agente a quem se pede "deixe mais rápido", sem esse diagnóstico, tende a atacar o que parece pesado: trocar o `solve_banded` por um solver na GPU, por exemplo. Pela lei de Amdahl, mesmo um `solve_banded` infinitamente rápido daria o ganho da última linha acima. O gargalo é a montagem da jacobiana, e aqui ela nem precisa ser calculada por diferenças: a jacobiana de $\theta'' + \delta e^\theta$ é conhecida ($-2/h^2 + \delta e^{\theta_i}$ na diagonal e $1/h^2$ fora dela), e é o que as versões da seção 6 usam. A função do pacote é genérica (serve para qualquer $f$), e paga por isso.

## Resumo

| Técnica | Ganho típico | Quando usar | Cuidado |
| --- | --- | --- | --- |
| melhor algoritmo (complexidade menor) | fatores de $10$ a $10^6$ | sempre primeiro | — |
| vetorização com NumPy | 10 a 100× sobre laços em Python | operações sobre arranjos inteiros | memória; vetores pequenos não pagam o custo fixo |
| processamento em lote | 10 a 1 000× | muitos problemas pequenos independentes | laços sequenciais continuam |
| compilação (Numba) | 10 a 100× sobre Python | laços com dependências, testes por elemento | compilação na primeira chamada; subconjunto do Python |
| GPU (PyTorch) | 10 a 100× sobre CPU, nos casos certos | operações grandes, dados residentes | `float32` exige rever tolerâncias; cópias; sincronizar ao medir |
| medir (`cronometrar`, `cProfile`) | — | antes e depois de toda otimização | mediana, aquecimento, mesmos dados |

## Exercícios

**Conceituais (sem computador)**

1. Estime o tempo para fatorar uma matriz densa de ordem $10^4$ num processador que faz $10^{11}$ operações por segundo, e a memória necessária em `float64` e em `float32`.
2. Um código gasta 70 % do tempo num núcleo que pode ser acelerado 20 vezes na GPU, e 30 % em pré e pós-processamento na CPU. Qual é o ganho total? E se a cópia dos dados para a GPU acrescentar 10 % ao tempo original?
3. Por que somar 1 000 vetores de 10 elementos com o NumPy é muito mais lento que somar um vetor de 10 000 elementos?
4. Em `float32`, qual é a menor tolerância **relativa** razoável para o critério de parada de Newton? E para a tolerância de um integrador de EDOs?

**Verificação (com agente)**

5. Peça a um agente para vetorizar o cálculo da temperatura do espalhador do capítulo 16 (diferenças finitas explícitas) em PyTorch, em `float32` e em `float64`. Compare a temperatura máxima depois de 60 s nas duas precisões e meça o tempo de cada uma, com a função `cronometrar`.
6. Peça a um agente uma versão em Numba do Monte Carlo do apêndice A que use `numba.prange` para dividir os cenários entre os núcleos do processador. Meça o ganho em função do número de núcleos e compare com a lei de Amdahl.
7. Peça a um agente para resolver o problema do poço termométrico (capítulo 14) para 10 mil combinações de comprimento e emissividade. Verifique a resposta da versão acelerada contra a da versão simples numa amostra de 100 combinações.

**Projeto aberto**

8. **Quanto custa a resposta?** Escolha um problema de um dos capítulos 12 a 16 e produza, para ele, uma curva de "erro contra tempo de execução" para pelo menos três implementações (por exemplo, NumPy, Numba e PyTorch, e dois métodos numéricos). Qual combinação atinge um erro de $10^{-6}$ no menor tempo? A resposta muda com o tamanho do problema?

## Leituras

- M. Gorelick e I. Ozsvald, *High Performance Python*, 2ª ed., O'Reilly, 2020.
- J. VanderPlas, *Python Data Science Handbook*, O'Reilly, 2016, capítulo 2 (cálculos com NumPy).
- Documentação do Numba ("A ~5 minute guide to Numba") e do PyTorch ("CUDA semantics", sobre execução assíncrona e sincronização).
- N. J. Higham, *Accuracy and Stability of Numerical Algorithms*, 2ª ed., SIAM, 2002 (precisão simples e dupla).